# 01 — Generate Synthetic Dataset

Runs the InfraGraph AI dataset generator to produce:
- PNG network-diagram images (1400 × 900)
- YOLO bounding-box labels
- Topology graph JSON
- Alert / RCA scenario JSON
- Preview contact sheets

In [ ]:
import subprocess, sys

NUM    = 300          # diagrams to generate
OUT    = '../data_generator/infragraph_dataset'
SEED   = 42

cmd = [
    sys.executable,
    '../data_generator/generate_infragraph_dataset.py',
    '--num',  str(NUM),
    '--out',  OUT,
    '--seed', str(SEED),
    '--annotated-preview',
    '--clean',
]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)

In [ ]:
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

sheet = Path(OUT) / 'previews' / 'contact_sheet.png'
if sheet.exists():
    plt.figure(figsize=(18, 12))
    plt.imshow(Image.open(sheet))
    plt.axis('off')
    plt.title('Contact sheet — first 20 diagrams', fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
bbox_sheet = Path(OUT) / 'previews' / 'bbox_contact_sheet.png'
if bbox_sheet.exists():
    plt.figure(figsize=(18, 12))
    plt.imshow(Image.open(bbox_sheet))
    plt.axis('off')
    plt.title('Annotated bounding boxes — first 20 diagrams', fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
import json
from collections import Counter

label_dir = Path(OUT) / 'labels'
class_names = ['router','switch','firewall','server','database','load_balancer','cloud_or_wan']
counts = Counter()

for lf in label_dir.rglob('*.txt'):
    for line in lf.read_text().splitlines():
        if line.strip():
            counts[int(line.split()[0])] += 1

fig, ax = plt.subplots(figsize=(10, 4))
names = [class_names[i] for i in sorted(counts)]
vals  = [counts[i] for i in sorted(counts)]
ax.bar(names, vals, color='steelblue')
ax.set_ylabel('Instance count')
ax.set_title('Object instances per class')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()